In [ ]:
import pandas as pd
from PIL import Image

In [ ]:
data_map = pd.read_csv(r'C:\Users\windows10\Documents\GitHub\X-Ray_Pneumonia\data_map.csv') 
data_map.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data_map['class_encoded'] = le.fit_transform(data_map['class'])

In [ ]:
data_map

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn, optim
from torchvision import transforms, models

In [ ]:
class XRayDataset(Dataset):
    def __init__(self, df_split, transform=None):
        self.df_split = df_split
        self.transform = transform
    def __len__(self):
        return len(self.df_split)
    def __getitem__(self, index):
        img_path = self.df_split.iloc[index]['image_path']
        image = Image.open(img_path).convert('L')
        label = self.df_split.iloc[index]['class_encoded']
        if self.transform:
            image_trans = self.transform(image)
        return image_trans, label


In [ ]:
train_dataset = XRayDataset(data_map[data_map["split"] == "train"], transform=transforms.ToTensor())
val_dataset = XRayDataset(data_map[data_map["split"] == "val"], transform=transforms.ToTensor())

In [ ]:
data_map.iloc[827]

In [ ]:
train_dataset[827]

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)  

# ResNet18


In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)


In [ ]:
model.conv1 = nn.Conv2d(1, 64, kernel_size=(7,7), stride=(2,2), padding=(3,3), bias=False)
model.fc = nn.Linear(in_features=512, out_features=2, bias=True)

In [ ]:
model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss = nn.CrossEntropyLoss()

In [ ]:
from tqdm import tqdm

In [ ]:
for epoch in range(1):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, total=len(train_loader)):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        pred = model(images)
        loss_val = loss(pred, labels)
        loss_val.backward()
        optimizer.step()
        running_loss += loss_val.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")    


In [ ]:
torch.save(model.state_dict(),'../models/xray_resnet18.pth')

In [ ]:
img_path = r'C:\Users\windows10\Documents\GitHub\X-Ray_Pneumonia\X_ray\chest_xray\val\NORMAL\NORMAL2-IM-1427-0001.jpeg'
image = Image.open(img_path).convert('L')
img_trans = transforms.ToTensor()(image).unsqueeze(0).to(device)
pred = model(img_trans)
pred_class = torch.argmax(pred, dim=1).item()
le.inverse_transform([pred_class])

# LeNet


In [ ]:

le_model = models.GoogLeNet(weights=models.GoogLeNet_Weights.IMAGENET1K_V1)
le_model.conv1 = nn.Conv2d(1, 64, kernel_size=(7